In [0]:
# HIGH_GARDEN_CATALOG_PARAMETER
dbutils.widgets.text("catalog", "high_garden")
catalog = dbutils.widgets.get("catalog").strip() or "high_garden"
print(f"Using Unity Catalog: {catalog}")


In [0]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from pyspark.sql import functions as F

In [0]:
SOURCE_TABLE = (
    f"{catalog}.gold.forecasting_features"
)

OUTPUT_TABLE = (
    f"{catalog}.gold.market_anomalies"
)

EXPERIMENT_NAME = (
    "/Shared/high-garden-coffee"
)

In [0]:
sdf = spark.table(
    SOURCE_TABLE
)

pdf = sdf.toPandas()

print(
    pdf.shape
)

In [0]:
pdf[
    "relative_change"
] = np.where(
    pdf["lag_1"] > 0,

    (
        pdf["target"]
        - pdf["lag_1"]
    )
    /
    pdf["lag_1"],

    np.nan
)

In [0]:
pdf[
    "rolling_deviation"
] = np.where(
    pdf["rolling_std_3"] > 0,

    (
        pdf["target"]
        - pdf["rolling_mean_3"]
    )
    /
    pdf["rolling_std_3"],

    0.0
)

In [0]:
pdf[
    "relative_to_mean"
] = np.where(
    pdf["rolling_mean_3"] > 0,

    (
        pdf["target"]
        - pdf["rolling_mean_3"]
    )
    /
    pdf["rolling_mean_3"],

    np.nan
)

In [0]:
anomaly_features = [
    "relative_change",
    "rolling_deviation",
    "relative_to_mean",
    "rolling_std_3",
]

In [0]:
anomaly_df = (
    pdf[
        [
            "country",
            "coffee_type",
            "crop_year",
            "start_year",
            "target",
        ]
        +
        anomaly_features
    ]
    .copy()
)

In [0]:
anomaly_df = (
    anomaly_df
    .dropna(
        subset=[
            "rolling_std_3",
            "target",
        ]
    )
    .copy()
)

print(
    "Rows for anomaly detection:",
    len(anomaly_df)
)

In [0]:
anomaly_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "isolation_forest",
            IsolationForest(
                n_estimators=300,
                contamination=0.05,
                random_state=42,
            )
        ),
    ]
)

In [0]:
X_anomaly = (
    anomaly_df[
        anomaly_features
    ]
)

anomaly_pipeline.fit(
    X_anomaly
)

In [0]:
raw_labels = (
    anomaly_pipeline.predict(
        X_anomaly
    )
)

In [0]:
anomaly_df[
    "is_anomaly"
] = (
    raw_labels == -1
).astype(int)

In [0]:
model_step = (
    anomaly_pipeline
    .named_steps[
        "isolation_forest"
    ]
)

processed_X = (
    anomaly_pipeline[
        :-1
    ].transform(
        X_anomaly
    )
)

anomaly_df[
    "anomaly_score"
] = -model_step.score_samples(
    processed_X
)

In [0]:
anomaly_count = int(
    anomaly_df[
        "is_anomaly"
    ].sum()
)

anomaly_rate = (
    anomaly_count
    /
    len(anomaly_df)
)

print(
    "Anomalies:",
    anomaly_count
)

print(
    "Anomaly rate:",
    anomaly_rate
)

In [0]:
display(
    anomaly_df[
        anomaly_df[
            "is_anomaly"
        ] == 1
    ]
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(30)
)

In [0]:
assert (
    anomaly_df[
        "is_anomaly"
    ]
    .isin(
        [0, 1]
    )
    .all()
)

assert (
    anomaly_df[
        "anomaly_score"
    ]
    .isna()
    .sum()
    == 0
)

print(
    "Anomaly detection validation passed."
)

In [0]:
mlflow.set_experiment(
    EXPERIMENT_NAME
)

In [0]:
with mlflow.start_run(
    run_name="isolation_forest_market_anomalies"
) as run:

    mlflow.log_param(
        "algorithm",
        "IsolationForest"
    )

    mlflow.log_param(
        "n_estimators",
        300
    )

    mlflow.log_param(
        "contamination",
        0.05
    )

    mlflow.log_param(
        "features",
        ",".join(
            anomaly_features
        )
    )

    mlflow.log_metric(
        "anomaly_count",
        anomaly_count
    )

    mlflow.log_metric(
        "anomaly_rate",
        anomaly_rate
    )

    mlflow.sklearn.log_model(
        anomaly_pipeline,
        name="model"
    )

    mlflow.set_tag(
        "task",
        "anomaly_detection"
    )

    anomaly_run_id = (
        run.info.run_id
    )

print(
    "Anomaly MLflow run:",
    anomaly_run_id
)

In [0]:
anomaly_output = anomaly_df[
    [
        "country",
        "coffee_type",
        "crop_year",
        "start_year",
        "target",
        "relative_change",
        "rolling_deviation",
        "relative_to_mean",
        "anomaly_score",
        "is_anomaly",
    ]
].copy()

In [0]:
anomaly_sdf = (
    spark.createDataFrame(
        anomaly_output
    )
)

In [0]:
(
    anomaly_sdf.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        OUTPUT_TABLE
    )
)

In [0]:
display(spark.sql(f"""
SELECT
    country,
    crop_year,
    target,
    ROUND(relative_change * 100, 2)
        AS yoy_change_pct,
    ROUND(rolling_deviation, 2)
        AS rolling_deviation,
    ROUND(anomaly_score, 3)
        AS anomaly_score
FROM {catalog}.gold.market_anomalies
WHERE is_anomaly = 1
ORDER BY anomaly_score DESC;
"""))
